# Assuming we have a dataframe with X,y where:
## X=[text1,text2,...]
## y=[label1, label2,...]

Pipe (requirement: all transformers need to adhere to sklearn api):
* remove stopwords
* tokenize 
* vectorize
* (kMeans, topic modelling etc as augmentation for regularisation)
* classify with logistic regression or randomforest

! Remove documents that are purely administrative !

# metrics 

F1, Spec, Recall, NPV, Prec, AUROC 

In [ ]:
import pandas as pd

import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

import string

import re

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import matplotlib.pyplot as plt

from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

import numpy as np

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

from sklearn.metrics import accuracy_score, \
    roc_auc_score, \
    f1_score, \
    recall_score, \
    precision_score, \
    confusion_matrix

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
RS = 42

K = 10

LOW_THRESHOLD = 20

In [ ]:
df_consults_text = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")
df_consults_text = df_consults_text.drop(columns='date')

df_afsprak = pd.read_sas(r'L:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805\afspraken_20240805.sas7bdat', encoding='latin1')
df_afsprak.studyId_0831 = df_afsprak.studyId_0831.astype('uint64')

df_afsprak = df_afsprak.drop(['AfspCode', 'Agenda', 'period_start',\
    'period_end', 'Spec_Code', 'Specialisme', 'AgendaNaam', 'AgendaComment',\
    'AgendaKostenplaats', 'Description', 'AfspraakNummer', 'SubAgenda',\
    'serviceProv_Org_value', 'AfspraakStatus', 'status_code_original',\
    'status_display_original', 'type1_code', 'type1_display',\
    'type1_system', 'type2_code', 'type2_display', 'type2_system',\
    'type3_code', 'type3_display', 'type3_system', 'index_date'],axis='columns')

df_afsprak = df_afsprak.groupby('studyId_0831', sort=True).agg(lambda x: x.to_list())

In [ ]:
def isEligible(x:pd.Series):
    if 'NFALG' in x or 'NFGEH' in x or 'NFVMI+' in x or 'NFVAL+' in x:
        return 1
    else:
        return 0

In [ ]:
df_afsprak['label'] = df_afsprak.index_AfspCode.apply(isEligible)
df_afsprak = df_afsprak[['label']]

print(f'Number of patients: {df_afsprak.shape[0]}')

In [ ]:
df_merged = df_consults_text.merge(df_afsprak, on='studyId_0831')

df_merged = df_merged.drop_duplicates(subset=['studyId_0831', 'text'])

print(df_merged['label'].value_counts(normalize=True) * 100)

In [ ]:
# Funzione per contare il numero di parole in una stringa
def conta_parole(text):
    # Conta le parole separando per spazio
    return len(text.split())

In [ ]:
# Aggiungi una colonna con il conteggio delle parole
df_merged['words'] = df_merged['text'].apply(conta_parole)

# Filtra i campioni che hanno un numero di parole >= soglia_parole
df_filt = df_merged[df_merged['words'] >= LOW_THRESHOLD]

df_filt = df_filt.drop(columns='words')

df_filt = df_filt.reset_index(drop=True)

print(df_filt['label'].value_counts(normalize=True) * 100)

### CountVectorizer

In [ ]:
CountVectorizerKwargs = {
    'lowercase': True,
    'stop_words': stopwords.words('dutch'),
    'ngram_range': (1, 3),
    'min_df': 0.01,
    'max_df': 0.99
}

XGBClassifierKwargs = {
    'n_estimators': 100,
    'objective': 'binary:logistic'
}

In [ ]:
X = df_filt.iloc[:, :-1]
X = X.drop(columns=['studyId_0831'])

y = df_filt.iloc[:, -1]

groups = df_filt['studyId_0831']

splitter = StratifiedGroupKFold(
    n_splits = K,
    shuffle=True,
    random_state=RS
)

# sklearn compatible classes
#StopwordRemover = FunctionTransformer(lambda x: re_remove.sub(" ", x))# use FunctionTransformer
Vectorizer = CountVectorizer(**CountVectorizerKwargs)
Classifier = XGBClassifier(**XGBClassifierKwargs)

le_pipe = Pipeline([
    ("Remove stopwords & Vectorize", Vectorizer),
    ("Classifier", Classifier)
])

metrics_list = []

for fold, (train_idxs, test_idxs) in enumerate(splitter.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idxs], X.iloc[test_idxs]
    y_train, y_test = y[train_idxs], y[test_idxs]
    
    le_pipe.fit(X_train['text'], y_train)
    
    test_pred = le_pipe.predict(X_test['text'])
    
    accuracy = accuracy_score(y_test, test_pred)
    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)
    auc = roc_auc_score(y_test, test_pred)
    
    # Calcola la matrice di confusione
    tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
    
    # Calcola la Specificity
    specificity = tn / (tn + fp)
    
    # Aggiungi i risultati al dizionario (una riga per ogni fold)
    metrics_list.append({
        'fold': fold + 1,  # I numeri dei fold partono da 1
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    })

# Crea un DataFrame dai risultati raccolti
df_metrics = pd.DataFrame(metrics_list)

df_metrics = df_metrics.set_index('fold', drop=True)

In [ ]:
df_metrics_mean = df_metrics.mean()

df_metrics_mean

### TfidfVectorizer

In [ ]:
TfidfVectorizerKwargs = {
    'lowercase': True,
    'stop_words': stopwords.words('dutch'),
    'ngram_range': (1, 3),
    'min_df': 0.01,
    'max_df': 0.99
}

XGBClassifierKwargs = {
    'n_estimators': 100,
    'objective': 'binary:logistic'
}

In [ ]:
X = df_filt.iloc[:, :-1]
X = X.drop(columns=['studyId_0831'])

y = df_filt.iloc[:, -1]

groups = df_filt['studyId_0831']

splitter = StratifiedGroupKFold(
    n_splits = K,
    shuffle=True,
    random_state=RS
)

# sklearn compatible classes
#StopwordRemover = FunctionTransformer(lambda x: re_remove.sub(" ", x))# use FunctionTransformer
Vectorizer = TfidfVectorizer(**TfidfVectorizerKwargs)
Classifier = XGBClassifier(**XGBClassifierKwargs)

le_pipe = Pipeline([
    ("Remove stopwords & Vectorize", Vectorizer),
    ("Classifier", Classifier)
])

metrics_list = []

for fold, (train_idxs, test_idxs) in enumerate(splitter.split(X, y, groups)):
    X_train, X_test = X.iloc[train_idxs], X.iloc[test_idxs]
    y_train, y_test = y[train_idxs], y[test_idxs]
    
    le_pipe.fit(X_train['text'], y_train)
    
    test_pred = le_pipe.predict(X_test['text'])
    
    accuracy = accuracy_score(y_test, test_pred)
    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    f1 = f1_score(y_test, test_pred)
    auc = roc_auc_score(y_test, test_pred)
    
    # Calcola la matrice di confusione
    tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
    
    # Calcola la Specificity
    specificity = tn / (tn + fp)
    
    # Aggiungi i risultati al dizionario (una riga per ogni fold)
    metrics_list.append({
        'fold': fold + 1,  # I numeri dei fold partono da 1
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    })

# Crea un DataFrame dai risultati raccolti
df_metrics = pd.DataFrame(metrics_list)

df_metrics = df_metrics.set_index('fold', drop=True)

In [ ]:
df_metrics_mean = df_metrics.mean()

df_metrics_mean